In [ ]:
import sys
import warnings
from pathlib import Path

from analysis.aggregations import agg_contribution_score_by_role_relative

REPO_ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

import analysis
print(analysis.__file__)

from analysis import *
from analysis.aggregations import *
from analysis.plots import *
from analysis.uuid_extractor import *
from analysis.transform import forward_fill_users

## Jupyter notes

## Multiple runs: Load and transform run

In [ ]:
data_dir = REPO_ROOT / "data" / "runs" / "experiments"
# data_dir = REPO_ROOT / "data" / "runs" / "sample"

output_dir = REPO_ROOT / "data" / "output_graphs"

# Optional: set a timestamp prefix to load only matching runs, e.g. "26-02-26"
# Leave as None to load all runs under experimentData/
PREFIX = "full_system_mnist141214"

runs = load_runs(data_dir, prefix=PREFIX)

guids = extract_uuids_from_filenames(runs)
print(guids)

# Normalize units: wei → ETH, ratio → %
runs = normalize_runs(runs)
res = merge_runs(runs)

print(f"Loaded {len(runs)} run(s)")

if (len(runs) == 0):
    raise ValueError("No runs loaded. Check the data directory and prefix.")

# Plots

## GRS by user graphs
### Single experiment for all users

In [ ]:
data = res['users']
data = data[data['experiment_id'] == data['experiment_id'].iloc[0]]

vals = grs_by_user(data)
fig = plot_grs_by_user(vals, metadata=res['metadata'])

### Agg. GRS by role

In [ ]:
data = res['users']
aggregated = agg_grs_by_role(data, res['metadata'])
fig = plot_grs_by_role(aggregated)

# Per activation round
# Check why mal/fr falls before round 0 (since activation)

### Agg. GRS by role by aggregation rule (with disqualified and exited counts)

In [ ]:
data = res['users']
metadata = res['metadata']

# unpack dict like this: fig = grs_and_disqualified_dict['FedAVG']
grs_and_disqualified_dict = plot_grs_by_role_by_aggregation_rule(data, metadata, res)


## Model Performance 
### Global Accuracy for aggregation strategies

In [ ]:
data = res['global']

# It logs self.pytorch_model.accuracy[-1] — so it's the accuracy of the global model evaluated after
# merging all participants' weights each round. That's the true global model performance, distinct from
# individual user accuracies.

# vals = grs_by_user(data)
# data[['round', 'global_accuracy']]
# data[vals['user_id'].isin([4, 5])].sort_values(['user_id', 'round'])
between_low = 0
between_high = 10
data = data[data['round'].between(between_low, between_high)]

vals = global_acc_by_aggregation_strategy(data, res['metadata'])

fig_accuracy = plot_global_acc_by_aggregation_strategy(vals)

### Global Loss for aggregation strategies

In [ ]:
data = res['global']

# ●It logs self.pytorch_model.loss[-1] — so it's the loss of the global model evaluated after
#  merging all participants' weights each round. That's the true global model performance, distinct from
#  individual user accuracies.

# vals = grs_by_user(data)
# data[['round', 'global_loss']]
# data[vals['user_id'].isin([4, 5])].sort_values(['user_id', 'round'])

# data[['round', 'role', 'behavior', 'round_reputation_assigned', 'merged']]

data = data[data['round'].between(between_low, between_high)] #defined in accuracy cell above

vals = global_loss_by_aggregation_strategy(data, res['metadata'])
fig_loss = plot_global_loss_by_aggregation_strategy(vals)

### Accuracy and Loss over rounds for all runs (aggregated) (not pr aggregation strat) 

In [ ]:
data = res['global']
aggregated = agg_global_accuracy_loss_by_round(data)
fig = plot_accuracy_loss_over_rounds(aggregated) # Assign since it otherwise will plot twice.

### Plot: plot_gas_cost_by_tx_type

In [ ]:
# plot_round_kicked_by_strategy(aggregated)
data = res['receipts']
aggregated = agg_gas_used_by_tx_type(data, res['metadata'])
fig_gas = plot_gas_cost_by_tx_type(aggregated)

## Plot: Contribution score by role (relative)

In [ ]:
u = res['users']
c = res['contributions']
agg = agg_contribution_score_by_role_relative(u, c, res['metadata'])

fig = plot_contribution_score_by_role_relative(agg)

# Per activation round

## Query: Runtime warnings

In [ ]:
data = res['warnings']
data

## Merge weights by behavior

In [ ]:
users_data = res['users']

metadata = res['metadata'][['experiment_id', 'aggregation_rule']]
users_data = users_data.merge(metadata, how='left', on='experiment_id')

total_runs = users_data['experiment_id'].nunique()
# users_data[users_data['merged'] == True].groupby(['aggregation_rule', 'behavior']).size() / total_runs
# users_data[users_data['merged'] == False].groupby(['aggregation_rule', 'behavior']).size()

agg_weights = agg_merge_weights_by_behavior(users_data)
agg_stats   = agg_merge_stats_by_behavior(users_data)
# # #
# # # # plot
fig = plot_merge_weights_by_behavior(agg_weights, agg_stats)



## How to save a Graph:

In [ ]:
# save_figure creates a directory named experiment_name under data/output_graphs/ and saves the graph there
# experiment_name is a name used to give a descriptive name of the experiment the graph shows data for.
# Name of plotting method is used for the file name of the graph
# Suffix is optional, but can be used to add a specific label to the graph's file name (e.g. 0.1-Noise-Malicious)

# How to save a graph
# save_figure(fig_accuracy, output_dir, experiment_name=experiment_name, suffix="accuracy") 
# How to save a graph from dict
# save_figure(example_dict[key], output_dir, experiment_name=key, suffix="example") 



# experiment_name = PREFIX
# 
# save_figure(fig_accuracy, output_dir, experiment_name=experiment_name, suffix="accuracy") # How to save a graph
# save_figure(fig_loss, output_dir, experiment_name=experiment_name, suffix="loss")
# for rule in grs_and_disqualified_dict: # save all graphs in the dict with their respective rule as suffix
#     save_figure(grs_and_disqualified_dict[rule], output_dir, experiment_name=experiment_name, suffix=rule)


## How to delete a graph (From the mappings.txt)

In [ ]:
# Delete a saved graph and remove its entries from mappings.txt
# Copy the name from the directory and paste as experiment name.
# Copy the graph_id from the svg file and paste as graph_id
# Delete the last graph in an experiment removes the mappings.txt

# experiment_name = "test"
# graph_id = "001"
# delete_figure(output_dir / experiment_name, graph_id=graph_id)